In [17]:
import os
import importlib.util

print("현재 작업 폴더:", os.getcwd())

spec = importlib.util.find_spec("matplotlib")
print("matplotlib spec:", spec)
print("origin:", spec.origin if spec else "없음")

현재 작업 폴더: c:\python_course\final_project\eunbi
matplotlib spec: ModuleSpec(name='matplotlib', loader=<_frozen_importlib_external.SourceFileLoader object at 0x0000020E9F61F950>, origin='c:\\python_course\\final_project\\.venv\\Lib\\site-packages\\matplotlib\\__init__.py', submodule_search_locations=['c:\\python_course\\final_project\\.venv\\Lib\\site-packages\\matplotlib'])
origin: c:\python_course\final_project\.venv\Lib\site-packages\matplotlib\__init__.py


In [14]:
%pip install matplotlib seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys
print(sys.executable)

import matplotlib
print("matplotlib:", matplotlib.__version__)

import seaborn
print("seaborn:", seaborn.__version__)

c:\python_course\final_project\.venv\Scripts\python.exe
matplotlib: 3.10.8
seaborn: 0.13.2


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [6]:
folder_path = "C:/python_course/final_project/eunbi" 
trans_df = pd.read_csv(folder_path+"/성남시_실거래가_통합_매매__202604201615.csv", encoding='euc-kr')
pub_df = pd.read_csv(folder_path+"/성남시_공시지가_통합__202604221844.csv")

# 거래량 데이터

## 결측치
- 컬럼 '도로명': 결측 확인 -- 162개(1.72%) <결측 5% 미만: 제거 또는 단순 보정 결정>
- 법정동으로 분석할 경우 도로명, 지번 정보 필요한가 의문

In [2]:
trans_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9420 entries, 0 to 9419
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   시군구      9420 non-null   str  
 1   유형       9420 non-null   str  
 2   도로명      9258 non-null   str  
 3   용도지역     9420 non-null   str  
 4   지번       9420 non-null   str  
 5   거래금액     9420 non-null   int64
 6   해제사유발생일  9420 non-null   str  
 7   계약날짜     9420 non-null   str  
 8   동별거래량    9420 non-null   int64
dtypes: int64(2), str(7)
memory usage: 662.5 KB


In [3]:
trans_df['도로명'].isna().sum()

np.int64(162)

In [4]:
trans_df['도로명'].isna().mean()*100

np.float64(1.7197452229299364)

In [8]:
trans_df[trans_df['도로명'].isna()]

,시군구,유형,도로명,용도지역,지번,거래금액,해제사유발생일,계약날짜,동별거래량
24,경기도 성남시 분당구 야탑동,집합,NaN,제3종일반주거,3**,66000,-,2023-01-02,1064
87,경기도 성남시 분당구 삼평동,집합,NaN,근린상업,7**,29000,-,2023-02-04,657
176,경기도 성남시 분당구 야탑동,집합,NaN,중심상업,3**,19000,-,2023-03-06,1064
340,경기도 성남시 분당구 수내동,집합,NaN,중심상업,1*,44000,-,2023-05-17,809
418,경기도 성남시 분당구 야탑동,집합,NaN,제3종일반주거,3**,18000,-,2023-06-20,1064
...,...,...,...,...,...,...,...,...,...
9157,경기도 성남시 중원구 성남동,집합,NaN,근린상업,4944,25000,-,2024-10-26,173
9232,경기도 성남시 중원구 상대원동,집합,NaN,용도미지정,5471,1858048,-,2024-12-16,114
9253,경기도 성남시 중원구 성남동,집합,NaN,근린상업,4944,127000,-,2025-02-27,173
9308,경기도 성남시 중원구 상대원동,일반,NaN,제3종일반주거,5***,82000,-,2025-06-07,114


## 용도 구분
- 상업용 부동산: 중심상업, 일반상업, 근린상업, 준주거 <br>
- 준주거의 경우, 법적으로는 "주거용"이나 1층 상가 + 주거 혼합 형태 많아 분석 목적에 따라 조건부 포함한다고 함. <포함 여부 결정>

In [9]:
trans_df['용도지역'].value_counts()

용도지역
중심상업       2446
준주거        2224
근린상업       1594
일반상업       1205
제3종일반주거    1122
제2종일반주거     375
보전녹지        245
자연녹지         70
기타           70
일반공업         36
제1종일반주거      15
용도미지정         9
개발제한구역        4
일반주거          3
제2종전용주거       2
Name: count, dtype: int64

## 이상치

### 1. 계약날짜 > 해제사유발생일 여부 확인 <없음>
- 보통 계약 이후 해제(계약날짜 <= 해제사유발생일)
- 계약날짜: 실제 매매 계약이 체결된 날
- 해제사유발생일: 계약이 해제된 사유가 발생한 날짜

In [11]:
trans_df.head(10)

,시군구,유형,도로명,용도지역,지번,거래금액,해제사유발생일,계약날짜,동별거래량
0,경기도 성남시 분당구 정자동,집합,불정로,제2종일반주거,2**,44000,-,2023-01-31,1176
1,경기도 성남시 분당구 정자동,집합,정자일로,중심상업,*,91500,-,2023-01-31,1176
2,경기도 성남시 분당구 정자동,집합,정자일로,중심상업,*,46000,-,2023-01-31,1176
3,경기도 성남시 분당구 정자동,집합,정자일로,중심상업,*,79500,-,2023-01-31,1176
4,경기도 성남시 분당구 정자동,집합,정자일로,중심상업,*,73000,-,2023-01-31,1176
5,경기도 성남시 분당구 정자동,집합,불정로,제2종일반주거,2**,26000,-,2023-01-31,1176
6,경기도 성남시 분당구 구미동,집합,미금로,근린상업,2**,69500,-,2023-01-26,893
7,경기도 성남시 분당구 금곡동,집합,성남대로171번길,중심상업,1**,69000,-,2023-01-26,746
8,경기도 성남시 분당구 구미동,집합,미금로,근린상업,2**,69500,20230215,2023-01-26,893
9,경기도 성남시 분당구 이매동,일반,이매로,보전녹지,2**,258700,-,2023-01-26,261


In [36]:
trans_df2 = trans_df.copy()

In [38]:
trans_df2['해제사유발생일'] = pd.to_datetime(trans_df2['해제사유발생일'], errors='coerce')

In [ ]:
trans_df2[trans_df2['해제사유발생일'].notna()]

,시군구,유형,도로명,용도지역,지번,거래금액,해제사유발생일,계약날짜,동별거래량
8,경기도 성남시 분당구 구미동,집합,미금로,근린상업,2**,69500,2023-02-15,2023-01-26,893
12,경기도 성남시 분당구 정자동,집합,정자일로,제3종일반주거,1**,25500,2023-02-08,2023-01-20,1176
46,경기도 성남시 분당구 서현동,집합,서현로210번길,중심상업,2**,10000,2023-04-12,2023-02-23,1473
73,경기도 성남시 분당구 수내동,집합,황새울로200번길,중심상업,1*,36000,2023-05-24,2023-02-13,809
74,경기도 성남시 분당구 수내동,집합,황새울로200번길,중심상업,1*,36000,2023-05-24,2023-02-13,809
...,...,...,...,...,...,...,...,...,...
9272,경기도 성남시 중원구 성남동,일반,광명로,일반상업,3***,332213,2025-06-10,2025-03-25,173
9279,경기도 성남시 중원구 상대원동,집합,둔촌대로,제2종일반주거,1-42,106500,2025-05-01,2025-04-22,114
9307,경기도 성남시 중원구 성남동,일반,광명로,일반상업,3***,332213,2025-06-12,2025-06-10,173
9342,경기도 성남시 중원구 성남동,집합,성남대로,일반상업,4169,17592,2025-10-15,2025-08-06,173


In [43]:
trans_df2[trans_df2['계약날짜'] > trans_df2['해제사유발생일']]

,시군구,유형,도로명,용도지역,지번,거래금액,해제사유발생일,계약날짜,동별거래량


해제사유발생일이 계약날짜 이전으로 기록된 데이터는 없음

### 2. 동별거래량: 0이거나 음수이면 이상치

최솟값: 1건, 최댓값: 1,473건

In [44]:
trans_df['동별거래량'].describe()

count    9420.000000
mean      812.892569
std       447.050064
min         1.000000
25%       361.000000
50%       809.000000
75%      1176.000000
max      1473.000000
Name: 동별거래량, dtype: float64

### 3. 거래금액
- 전체/상업용 부동산: 최솟값_200만원(2,000,000원), 최댓값_1조9820억4140만원(1,982,041,400,000원)
- 상업용 부동산 매매금액이 200만원은 너무 작음. 보수적으로 접근한다 해도 최소 1,000만원 ~ 1,500만원부터 시작
- 이를 SQL로 거래량 집계할 때 고려했나

In [46]:
pd.set_option('display.float_format', '{:.0f}'.format)
trans_df['거래금액'].describe()

count        9420
mean       323492
std       5696715
min           200
25%         15575
50%         37000
75%         70025
max     198204140
Name: 거래금액, dtype: float64

In [51]:
commercial_df = trans_df[trans_df['용도지역'].isin(['중심상업', '일반상업', '근린상업', '준주거'])]
commercial_df['거래금액'].describe()

count        7469
mean       387950
std       6394651
min           200
25%         17500
50%         40000
75%         73500
max     198204140
Name: 거래금액, dtype: float64

In [53]:
commercial_df[commercial_df['거래금액'] == 200]

,시군구,유형,도로명,용도지역,지번,거래금액,해제사유발생일,계약날짜,동별거래량
655,경기도 성남시 분당구 야탑동,집합,야탑로,근린상업,5**,200,-,2023-09-22,1064
8964,경기도 성남시 중원구 성남동,집합,성남대로1147번길,일반상업,4***,200,-,2023-05-02,173


In [58]:
commercial_df[(commercial_df['계약날짜'] == '2023-05-02')]

,시군구,유형,도로명,용도지역,지번,거래금액,해제사유발생일,계약날짜,동별거래량
377,경기도 성남시 분당구 서현동,집합,서현로210번길,중심상업,2**,31000,-,2023-05-02,1473
378,경기도 성남시 분당구 구미동,집합,탄천상로,일반상업,1*,20000,-,2023-05-02,893
379,경기도 성남시 분당구 구미동,집합,탄천상로,일반상업,1*,20000,20230713,2023-05-02,893
8964,경기도 성남시 중원구 성남동,집합,성남대로1147번길,일반상업,4***,200,-,2023-05-02,173


In [59]:
commercial_df[(commercial_df['시군구'] == '경기도 성남시 분당구 야탑동') & (commercial_df['계약날짜'] == '2025-06-19')]

,시군구,유형,도로명,용도지역,지번,거래금액,해제사유발생일,계약날짜,동별거래량
4142,경기도 성남시 분당구 야탑동,집합,야탑로,일반상업,367-4,800,-,2025-06-19,1064
4143,경기도 성남시 분당구 야탑동,집합,성남대로916번길,일반상업,366-3,48000,-,2025-06-19,1064
4148,경기도 성남시 분당구 야탑동,집합,야탑로,일반상업,367-4,800,-,2025-06-19,1064
4149,경기도 성남시 분당구 야탑동,집합,성남대로916번길,일반상업,366-3,48000,-,2025-06-19,1064
4154,경기도 성남시 분당구 야탑동,집합,야탑로,일반상업,367-4,800,-,2025-06-19,1064
4155,경기도 성남시 분당구 야탑동,집합,성남대로916번길,일반상업,366-3,48000,-,2025-06-19,1064
4160,경기도 성남시 분당구 야탑동,집합,야탑로,일반상업,367-4,800,-,2025-06-19,1064
4161,경기도 성남시 분당구 야탑동,집합,성남대로916번길,일반상업,366-3,48000,-,2025-06-19,1064
4166,경기도 성남시 분당구 야탑동,집합,야탑로,일반상업,367-4,800,-,2025-06-19,1064
4167,경기도 성남시 분당구 야탑동,집합,성남대로916번길,일반상업,366-3,48000,-,2025-06-19,1064


In [56]:
commercial_df.sort_values(by='거래금액').head(20)

,시군구,유형,도로명,용도지역,지번,거래금액,해제사유발생일,계약날짜,동별거래량
8964,경기도 성남시 중원구 성남동,집합,성남대로1147번길,일반상업,4***,200,-,2023-05-02,173
655,경기도 성남시 분당구 야탑동,집합,야탑로,근린상업,5**,200,-,2023-09-22,1064
940,경기도 성남시 분당구 이매동,집합,양현로94번길,근린상업,1**,500,-,2023-12-26,261
9032,경기도 성남시 중원구 성남동,집합,성남대로1147번길,일반상업,4***,500,-,2023-11-20,173
4142,경기도 성남시 분당구 야탑동,집합,야탑로,일반상업,367-4,800,-,2025-06-19,1064
4148,경기도 성남시 분당구 야탑동,집합,야탑로,일반상업,367-4,800,-,2025-06-19,1064
4166,경기도 성남시 분당구 야탑동,집합,야탑로,일반상업,367-4,800,-,2025-06-19,1064
4154,경기도 성남시 분당구 야탑동,집합,야탑로,일반상업,367-4,800,-,2025-06-19,1064
4172,경기도 성남시 분당구 야탑동,집합,야탑로,일반상업,367-4,800,-,2025-06-19,1064
4178,경기도 성남시 분당구 야탑동,집합,야탑로,일반상업,367-4,800,-,2025-06-19,1064


In [62]:
commercial_df[commercial_df.duplicated(subset=['시군구', '계약날짜'], keep=False)]

,시군구,유형,도로명,용도지역,지번,거래금액,해제사유발생일,계약날짜,동별거래량
1,경기도 성남시 분당구 정자동,집합,정자일로,중심상업,*,91500,-,2023-01-31,1176
2,경기도 성남시 분당구 정자동,집합,정자일로,중심상업,*,46000,-,2023-01-31,1176
3,경기도 성남시 분당구 정자동,집합,정자일로,중심상업,*,79500,-,2023-01-31,1176
4,경기도 성남시 분당구 정자동,집합,정자일로,중심상업,*,73000,-,2023-01-31,1176
6,경기도 성남시 분당구 구미동,집합,미금로,근린상업,2**,69500,-,2023-01-26,893
...,...,...,...,...,...,...,...,...,...
9398,경기도 성남시 중원구 여수동,집합,양현로,근린상업,190,37478,-,2025-11-20,31
9399,경기도 성남시 중원구 여수동,집합,양현로,근린상업,190,42022,-,2025-11-20,31
9410,경기도 성남시 중원구 도촌동,집합,도촌남로,준주거,567,193303,-,2025-12-11,43
9411,경기도 성남시 중원구 도촌동,집합,도촌남로,준주거,567,37138,-,2025-12-11,43


논의사항
- 동, 용도, 계약날짜 같은데 거래금액이 달라 여러 행으로 나뉘고 있음
- 거래량만 볼 건가, 거래금액만 볼 건가에 따라 SQL 활용 분석용 테이블 제작 작업 보완해야 할듯

# 공시지가 데이터

## 결측치 없음

In [63]:
pub_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15557 entries, 0 to 15556
Data columns (total 11 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   고유번호     15557 non-null  int64
 1   법정동코드    15557 non-null  int64
 2   법정동명     15557 non-null  str  
 3   특수지구분명   15557 non-null  str  
 4   지번       15557 non-null  str  
 5   기준연도     15557 non-null  int64
 6   기준월      15557 non-null  int64
 7   공시지가     15557 non-null  int64
 8   표준지여부    15557 non-null  str  
 9   기준년월     15557 non-null  str  
 10  건물용도분류명  15557 non-null  str  
dtypes: int64(5), str(6)
memory usage: 1.3 MB


## 중복행 없음

In [65]:
pub_df[pub_df.duplicated(subset=['고유번호', '법정동코드', '법정동명', '특수지구분명', '지번', '기준연도', '기준월', '공시지가', '표준지여부', '기준년월', '건물용도분류명'], keep=False)]

,고유번호,법정동코드,법정동명,특수지구분명,지번,기준연도,기준월,공시지가,표준지여부,기준년월,건물용도분류명


In [68]:
pub_df['건물용도분류명'].value_counts()

건물용도분류명
상업용    15557
Name: count, dtype: int64

In [67]:
pub_df['특수지구분명'].value_counts()

특수지구분명
일반    15542
산        15
Name: count, dtype: int64

산...?

## 공시지가
최솟값: 1m²당 2만3,500원(23,500원) <br>
최댓값: 1m²당 3,004만원(30,040,000원)

In [69]:
pub_df['공시지가'].describe()

count      15557
mean     5108037
std      2761173
min        23500
25%      3321000
50%      4507000
75%      6017000
max     30040000
Name: 공시지가, dtype: float64

In [79]:
pub_df['공시지가'].quantile([0.01, 0.99])

0     604412
1   14024400
Name: 공시지가, dtype: float64

In [11]:
q_1 = np.percentile(pub_df['공시지가'], q=25)
q_3 = np.percentile(pub_df['공시지가'], q=75)
iqr = q_3 - q_1

lower = q_1 - 1.5 * iqr
upper = q_3 + 1.5 * iqr

outliers = pub_df[
    (pub_df['공시지가'] < lower) |
    (pub_df['공시지가'] > upper)
]

outliers

,고유번호,법정동코드,법정동명,특수지구분명,지번,기준연도,기준월,공시지가,표준지여부,기준년월,건물용도분류명
108,4113511100101510000,4113511100,경기도 성남시 분당구 금곡동,일반,151,2023,1,10910000,Y,2023-01-01,상업용
109,4113511100101530000,4113511100,경기도 성남시 분당구 금곡동,일반,153,2023,1,10910000,N,2023-01-01,상업용
110,4113511100101550000,4113511100,경기도 성남시 분당구 금곡동,일반,155,2023,1,10910000,N,2023-01-01,상업용
111,4113511100101580000,4113511100,경기도 성남시 분당구 금곡동,일반,158,2023,1,11360000,N,2023-01-01,상업용
112,4113511100101590000,4113511100,경기도 성남시 분당구 금곡동,일반,159,2023,1,11360000,N,2023-01-01,상업용
...,...,...,...,...,...,...,...,...,...,...,...
15262,4113313200100950000,4113313200,경기도 성남시 중원구 중앙동,일반,95,2025,1,10780000,N,2025-01-01,상업용
15282,4113313200103070000,4113313200,경기도 성남시 중원구 중앙동,일반,307,2025,1,11550000,N,2025-01-01,상업용
15285,4113313200103350000,4113313200,경기도 성남시 중원구 중앙동,일반,335,2025,1,11200000,N,2025-01-01,상업용
15289,4113313200104210000,4113313200,경기도 성남시 중원구 중앙동,일반,421,2025,1,15100000,N,2025-01-01,상업용
